In [6]:
import sqlite3
import json
import csv
from cinesync.paths import DATA_DIR

FAILURE_LOG_PATH = DATA_DIR / "letterboxd_scrape_failures.jsonl"
IGNORE_CSV_PATH = DATA_DIR / "tmdb_ignore.csv"

failed_ids = set()
with open(FAILURE_LOG_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        failed_ids.add(entry["tmdb_id"])

print(f"{len(failed_ids)} unique tmdb_ids found in failure log.")

conn = sqlite3.Connection(DATA_DIR / "cinesync.db")

failed_ids_list = list(failed_ids)
rows = []
CHUNK = 500  # stay under SQLite's default ~999 bound-parameter limit

for i in range(0, len(failed_ids_list), CHUNK):
    chunk = failed_ids_list[i : i + CHUNK]
    placeholders = ",".join("?" * len(chunk))
    cursor = conn.execute(
        f"""
        SELECT tmdb_id, content_type, name
        FROM titles
        WHERE tmdb_id IN ({placeholders})
          AND content_type = 'movie'
        """,
        chunk,
    )
    rows.extend(cursor.fetchall())

conn.close()

found_ids = {r[0] for r in rows}
missing_ids = failed_ids - found_ids
if missing_ids:
    print(
        f"Note: {len(missing_ids)} failed tmdb_ids have no matching row in "
        f"titles (never made it past TMDB ingestion) — not included in CSV."
    )

with open(IGNORE_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["tmdb_id", "content_type", "title"])
    for tmdb_id, content_type, name in sorted(rows):
        writer.writerow([tmdb_id, content_type, name])

print(f"Wrote {len(rows)} candidate rows to {IGNORE_CSV_PATH}")

178 unique tmdb_ids found in failure log.
Wrote 178 candidate rows to /home/abdullah/Developer/CineSync/data/tmdb_ignore.csv
